<a href="https://colab.research.google.com/github/KaiIkuta/MIDSSAR/blob/main/MIDSSAR_reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Reduction code for MIDSSAR data**

Image (ds9 and sxiv) and Plot (gnuplot) are skipped.


[https://www.kusastro.kyoto-u.ac.jp/~iwamuro/MIDSSAR/index.html](https://www.kusastro.kyoto-u.ac.jp/~iwamuro/MIDSSAR/index.html)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Copy files for reduction to directory on observation date

In [ ]:
date = "20260921"

In [ ]:
!cp -r /content/drive/MyDrive/MIDSSAR/code/* /content/drive/MyDrive/MIDSSAR/{date}/
!ls -l /content/drive/MyDrive/MIDSSAR/{date}/

total 130279
drwx------ 2 root root     4096 Sep 22 03:27 20260921C
drwx------ 2 root root     4096 Sep 22 03:27 20260921H
-rw------- 1 root root 11062080 Sep 24 04:53 ampgC1_20260720.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgC1.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgC2_20260720.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgC2.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgC3_20260720.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgC3.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgH1_20260720.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgH1.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgH2_20260720.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgH2.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgH3_20260720.fits
-rw------- 1 root root 11062080 Sep 24 04:53 ampgH3.fits
-rw------- 1 root root    16640 Sep 22 03:26 ave
-rw------- 1 root root     4054 Sep 24 04:53 ave.c
-rw------- 1 root root     2998 Sep 24 

In [ ]:
%cd /content/drive/MyDrive/MIDSSAR/{date}/

/content/drive/MyDrive/MIDSSAR/20260921


Complile all C codes

In [ ]:
!gcc ave.c -o ave -lm -lcfitsio -Wall
!gcc mkflat.c -o mkflat -lm -lcfitsio -Wall
!gcc mzwarp.c -o mzwarp -lm -lcfitsio -Wall
!gcc sigclip.c -o sigclip -lm -lcfitsio -Wall
!gcc reduc.c -o reduc -lm -lcfitsio -Wall
!gcc double.c -o double -lm -lcfitsio -Wall
!gcc merge.c -o merge -lm -Wall

reduc.c: In function ‘main’:
reduc.c:469:28: warning: ‘%03d’ directive writing between 3 and 11 bytes into a region of size between 10 and 63 []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat-overflow=-Wformat-overflow=]8;;]
  469 |     else sprintf(fname,"%s_%03d.fits",fname1,k);
      |                            ^~~~
reduc.c:469:24: note: directive argument in the range [-2147483647, 2147483647]
  469 |     else sprintf(fname,"%s_%03d.fits",fname1,k);
      |                        ^~~~~~~~~~~~~~
reduc.c:469:10: note: ‘sprintf’ output between 10 and 71 bytes into a destination of size 64
  469 |     else sprintf(fname,"%s_%03d.fits",fname1,k);
      |          ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Make directories and move files

In [ ]:
!mkdir  {date}H
!mkdir  {date}C
!mv mzrh* {date}H
!mv mzrc* {date}C

In [ ]:
!echo 0 > {date}C/camnum
!echo 1 > {date}H/camnum

Move to H-alpha directory

In [ ]:
%cd ./{date}H

/content/drive/MyDrive/MIDSSAR/20260921/20260921H


Check observation log

In [ ]:
!cat mzrh.log

mzrh00069742.fits 2026-09-21 18:02:01.324496    1.00 x01 -20 -21.8  37.0  24.6  24.8  24.9  24.4   879.42   45.69   670  1246 18150 FOCUSING Shut FeNe Fan
mzrh00069743.fits 2026-09-21 18:02:37.450391    1.00 x01 -20 -22.0  37.6  24.6  24.9  24.8  24.4   893.30  714.09   665 65535 17600 FOCUSING FeNe Fan
mzrh00069744.fits 2026-09-21 18:02:40.609605    1.00 x01 -20 -22.0  37.5  24.6  24.9  24.9  24.4   894.24  710.12   680 65535 17700 FOCUSING FeNe Fan
mzrh00069745.fits 2026-09-21 18:02:43.908356    1.00 x01 -20 -22.0  37.5  24.6  24.9  24.8  24.4   893.27  705.48   674 65535 17800 FOCUSING FeNe Fan
mzrh00069746.fits 2026-09-21 18:02:47.188317    1.00 x01 -20 -22.0  37.5  24.6  24.9  24.8  24.4   895.01  702.57   671 65535 17900 FOCUSING FeNe Fan
mzrh00069747.fits 2026-09-21 18:02:50.492691    1.00 x01 -20 -22.0  37.5  24.7  24.9  24.9  24.4   897.98  701.62   660 65535 18000 FOCUSING FeNe Fan
mzrh00069748.fits 2026-09-21 18:02:53.945688    1.00 x01 -20 -22.0  37.5  24.7  25.0  24.8  24.

Check Bias files

In [ ]:
!grep BIAS mzrh.log | cat -n
!grep BIAS mzrh.log > bias

     1	mzrh00069774.fits 2026-09-21 18:04:22.332867    0.01 x16 -20 -21.8  36.9  24.9  25.2  24.8  24.3  1230.18   83.45   570  3444 18086 BIAS Shut Fan
     2	mzrh00069775.fits 2026-09-21 18:04:22.413296    0.01 x16 -20 -21.7  36.8  24.9  25.2  24.8  24.3  1231.47   83.10   578  3347 18086 BIAS Shut Fan
     3	mzrh00069776.fits 2026-09-21 18:04:22.463760    0.01 x16 -20 -21.7  36.8  24.9  25.2  24.8  24.3  1229.26   83.44   526  3410 18086 BIAS Shut Fan
     4	mzrh00069777.fits 2026-09-21 18:04:22.513788    0.01 x16 -20 -21.7  36.8  24.9  25.2  24.8  24.3  1228.89   83.27   559  3300 18086 BIAS Shut Fan
     5	mzrh00069778.fits 2026-09-21 18:04:22.564381    0.01 x16 -20 -21.7  36.8  24.9  25.2  24.8  24.3  1229.92   82.23   507  3427 18086 BIAS Shut Fan
     6	mzrh00069779.fits 2026-09-21 18:04:22.616633    0.01 x16 -20 -21.7  36.8  24.9  25.2  24.8  24.3  1229.15   82.42   575  3234 18086 BIAS Shut Fan
     7	mzrh00069780.fits 2026-09-21 18:04:22.666555    0.01 x16 -20 -21.7  36.8  2

Make master Bias

In [ ]:
!../sigclip bias bias.fits

Clipping threshold: 3 sigma
Read: bias ... 
Read: mzrh00069873.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 40.8828 ± 5.83354 (Clipped: 126736 pix)
Average: 1223.97 ± 53.3821 (Clipped: 55945 pix)
Append: sigclip.log ... OK
Write bias.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


Check Flat files

In [ ]:
!grep INSTFLAT mzrh.log | cat -n

     1	mzrh00069974.fits 2026-09-21 18:30:11.967445    0.02 x16 -20 -20.0  31.8  25.0  24.9  24.1  23.2  2691.71 2778.35   497 12660 14086 INSTFLAT Cont Fan
     2	mzrh00069975.fits 2026-09-21 18:30:12.034345    0.02 x16 -20 -19.9  31.8  25.0  24.9  24.1  23.2  2690.18 2777.59   486 12725 14086 INSTFLAT Cont Fan
     3	mzrh00069976.fits 2026-09-21 18:30:12.094066    0.02 x16 -20 -19.9  31.8  25.0  24.9  24.1  23.2  2689.00 2777.96   642 12848 14086 INSTFLAT Cont Fan
     4	mzrh00069977.fits 2026-09-21 18:30:12.153769    0.02 x16 -20 -19.9  31.8  25.0  24.9  24.1  23.2  2691.04 2778.96   586 12753 14086 INSTFLAT Cont Fan
     5	mzrh00069978.fits 2026-09-21 18:30:12.213491    0.02 x16 -20 -19.9  31.8  25.0  24.9  24.1  23.2  2689.61 2778.23   549 12479 14086 INSTFLAT Cont Fan
     6	mzrh00069979.fits 2026-09-21 18:30:12.273513    0.02 x16 -20 -19.9  31.8  25.0  24.9  24.1  23.2  2690.91 2778.66   475 13276 14086 INSTFLAT Cont Fan
     7	mzrh00069980.fits 2026-09-21 18:30:12.333194    0.0

In [ ]:
!grep INSTFLAT mzrh.log | sed -n '001,100p' > flat1
!grep INSTFLAT mzrh.log | sed -n '101,200p' > flat2
#!grep INSTFLAT mzrh.log | sed -n '201,300p' >> flat1
#!grep INSTFLAT mzrh.log | sed -n '301,400p' >> flat2
#!grep INSTFLAT mzrh.log | sed -n '401,500p' >> flat1
#!grep INSTFLAT mzrh.log | sed -n '501,600p' >> flat2

In [ ]:
!grep DARK mzrh.log | grep Cont | cat -n
!grep DARK mzrh.log | grep Cont > flatd

     1	mzrh00069874.fits 2026-09-21 18:30:00.460476    0.02 x16 -20 -20.0  31.8  25.0  25.0  24.2  23.2  1241.84   89.47   501  3506 18086 DARK Cont Fan
     2	mzrh00069875.fits 2026-09-21 18:30:00.550856    0.02 x16 -20 -20.0  31.8  25.0  25.0  24.2  23.2  1241.27   89.64   540  3511 18086 DARK Cont Fan
     3	mzrh00069876.fits 2026-09-21 18:30:00.611778    0.02 x16 -20 -20.0  31.8  25.0  25.0  24.2  23.2  1240.52   88.54   498  3493 18086 DARK Cont Fan
     4	mzrh00069877.fits 2026-09-21 18:30:00.673131    0.02 x16 -20 -20.0  31.8  25.0  25.0  24.2  23.2  1239.62   89.86   566  3452 18086 DARK Cont Fan
     5	mzrh00069878.fits 2026-09-21 18:30:00.735091    0.02 x16 -20 -20.0  31.8  25.0  25.0  24.2  23.2  1241.06   89.07   565  3569 18086 DARK Shut Cont Fan
     6	mzrh00069879.fits 2026-09-21 18:30:00.796355    0.02 x16 -20 -20.0  31.8  25.0  25.0  24.2  23.2  1238.38   88.91   535  3680 18086 DARK Shut Cont Fan
     7	mzrh00069880.fits 2026-09-21 18:30:00.857764    0.02 x16 -20 -20.

Make master Flat

In [ ]:
!../sigclip flatd flatd.fits

Clipping threshold: 3 sigma
Read: flatd ... 
Read: mzrh00069973.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 42.9751 ± 6.42422 (Clipped: 118779 pix)
Average: 1233.75 ± 56.8138 (Clipped: 65898 pix)
Append: sigclip.log ... OK
Write flatd.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


In [ ]:
!../sigclip flat1 flat1.fits flatd.fits
!../sigclip flat2 flat2.fits flatd.fits

Clipping threshold: 3 sigma
Dark image: flatd.fits
Read: flat1 ... 
Read: mzrh00070073.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read flatd.fits  ... 2160 x 1280 format Subtraction OK
Bright : [1:2160,485:770]
Center : [1:2161,527:727]
Sigma  : 311.471 ± 36.0617 (Clipped: 1141 pix)
Average: 6857.49 ± 1190.02 (Clipped: 46 pix)
Append: sigclip.log ... OK
Normalization OK
Write flat1.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Dark image: flatd.fits
Read: flat2 ... 
Read: mzrh00070173.fits ... OK
Maximum rejected pixel: [2160,1280] 1/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read flatd.fits  ... 2160 x 1280 format Subtraction OK
Bright : [1:2160,483:770]
Center : [1:2161,526:726]
Sigma  : 315.028 ± 36.0192 (Clipped: 1032 pix)
Average: 7012.68 ± 1186.17 (Clipped: 2 pix)
Append: sigclip.log ... OK
Normalization OK
Write flat2.fits ... 2160 x 1280 format O

In [ ]:
!../ave flat1.fits flat2.fits flat0.fits
!../mkflat flat0
!mv flat0_xy.fits flat.fits

Read: flat1.fits ... 2160 x 1280 format OK
Read: flat2.fits ... 2160 x 1280 format OK
Write flat0.fits ... 2160 x 1280 format OK
Read: flat0.fits ... 2160 x 1280 format OK
Bright : [1:2160,494:760]
Center : [80:2080,527:727]
 1.031208 0.969924 1.026566 0.966468
 0.978170 1.021625 0.977623 1.037305
 1.034978 0.963899 1.032117 0.966506
 0.970013 1.027755 0.970485 1.025357
Write flat0_xy.fits ... 2160 x 1280 format OK
Write flat0_res.fits ... 2160 x 1280 format OK


Check Skyflat files

In [ ]:
!pwd

/content/drive/MyDrive/MIDSSAR/20260921/20260921H


In [ ]:
!pwd

/content/drive/MyDrive/MIDSSAR/20260921/20260921H


In [ ]:
!cp -r /content/drive/MyDrive/MIDSSAR/data/20260922/20260922H/mzwarp_dxdy.fits /content/drive/MyDrive/MIDSSAR/20260921/20260921H/
!cp -r /content/drive/MyDrive/MIDSSAR/data/20260922/20260922H/flats.fits /content/drive/MyDrive/MIDSSAR/20260921/20260921H/
!cp -r /content/drive/MyDrive/MIDSSAR/data/20260922/20260922H/flat.dat /content/drive/MyDrive/MIDSSAR/20260921/20260921H/

In [ ]:
!grep SKYFLAT mzrh.log | cat -n

In [ ]:
!grep SKYFLAT mzrh.log | sed -n '101,200p' > sky
!grep DARK mzrh.log | grep -v Cont | grep -v FeNe | cat -n

     1	mzrh00070785.fits 2026-09-22 16:02:11.524767    0.50 x16 -20 -21.6  41.0  27.1  27.5  26.3  26.0  1247.20   90.28   535  4388 18046 DARK Shut Fan Cl2
     2	mzrh00070786.fits 2026-09-22 16:02:12.076298    0.50 x16 -20 -21.7  41.0  27.1  27.5  26.3  26.0  1245.13   91.25   575  3783 18046 DARK Shut Fan Cl2
     3	mzrh00070787.fits 2026-09-22 16:02:12.616303    0.50 x16 -20 -21.7  41.0  27.1  27.5  26.3  26.0  1246.93   90.96   630  4096 18046 DARK Shut Fan Cl2
     4	mzrh00070788.fits 2026-09-22 16:02:13.156117    0.50 x16 -20 -21.7  41.0  27.1  27.5  26.3  26.0  1245.51   91.12   542  4132 18046 DARK Shut Fan Cl2
     5	mzrh00070789.fits 2026-09-22 16:02:13.695844    0.50 x16 -20 -21.7  41.0  27.1  27.5  26.3  26.0  1246.89   91.05   529  4055 18046 DARK Shut Fan Cl2
     6	mzrh00070790.fits 2026-09-22 16:02:14.235756    0.50 x16 -20 -21.7  41.0  27.1  27.5  26.3  26.0  1248.21  105.76   608 65535 18046 DARK Shut Fan Cl2
     7	mzrh00070791.fits 2026-09-22 16:02:14.775484    0.5

In [ ]:
!grep DARK mzrh.log | grep -v Cont | grep -v FeNe | sed -n '11,20p' > skyd

Make master Skyflat

In [ ]:
!../sigclip skyd skyd.fits

Clipping threshold: 3 sigma
Read: skyd ... 
Read: mzrh00070904.fits ... OK
Maximum rejected pixel: [2160,1280] 0/10 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 39.5151 ± 11.1954 (Clipped: 49310 pix)
Average: 1240.94 ± 58.9388 (Clipped: 66027 pix)
Append: sigclip.log ... OK
Write skyd.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


In [ ]:
!grep DARK mzrh.log | grep Cont

mzrh00071137.fits 2026-09-22 18:37:42.367275    0.02 x16 -20 -21.7  34.8  23.8  24.0  24.2  23.6  1240.44   89.61   375  3508 17944 DARK Shut Cont Fan
mzrh00071138.fits 2026-09-22 18:37:42.456774    0.02 x16 -20 -21.6  34.6  23.9  24.0  24.2  23.6  1238.94   89.17   566  3393 17944 DARK Shut Cont Fan
mzrh00071139.fits 2026-09-22 18:37:42.516681    0.02 x16 -20 -21.6  34.6  23.9  24.0  24.2  23.6  1239.59   89.60   523  3501 17944 DARK Shut Cont Fan
mzrh00071140.fits 2026-09-22 18:37:42.577516    0.02 x16 -20 -21.6  34.6  23.9  24.0  24.2  23.6  1239.19   89.31   510  3469 17944 DARK Shut Cont Fan
mzrh00071141.fits 2026-09-22 18:37:42.637128    0.02 x16 -20 -21.6  34.6  23.9  24.0  24.2  23.6  1239.36   88.96   428  3447 17944 DARK Shut Cont Fan
mzrh00071142.fits 2026-09-22 18:37:42.697669    0.02 x16 -20 -21.6  34.6  23.9  24.0  24.2  23.6  1238.81   88.80   422  3646 17944 DARK Shut Cont Fan
mzrh00071143.fits 2026-09-22 18:37:42.757318    0.02 x16 -20 -21.6  34.6  23.9  24.0  24.2  23

In [ ]:
!grep BIAS mzrh.log | sed -n '1,100p' > bias_sky
!../sigclip bias_sky bias_sky.fits
!grep INSTFLAT mzrh.log | sed -n '001,100p' > flat1_sky
!grep INSTFLAT mzrh.log | sed -n '101,200p' > flat2_sky
#!grep INSTFLAT mzrh.log | sed -n '201,300p' >> flat1_sky
#!grep INSTFLAT mzrh.log | sed -n '301,400p' >> flat2_sky
!grep DARK mzrh.log | grep Cont | sed -n '1,200p'> flatd_sky
!../sigclip flatd_sky flatd_sky.fits
!../sigclip flat1_sky flat1_sky.fits flatd_sky.fits
!../sigclip flat2_sky flat2_sky.fits flatd_sky.fits
!../ave flat1_sky.fits flat2_sky.fits flat0_sky.fits
!../mkflat flat0_sky
!mv flat0_sky_xy.fits flat_sky.fits

Clipping threshold: 3 sigma
Read: bias_sky ... 
Read: mzrh00071136.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 40.8688 ± 5.84648 (Clipped: 126244 pix)
Average: 1221.51 ± 53.3062 (Clipped: 56205 pix)
Append: sigclip.log ... OK
Write bias_sky.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Read: flatd_sky ... 
Read: mzrh00071236.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 42.8872 ± 6.42501 (Clipped: 118513 pix)
Average: 1233.14 ± 56.7559 (Clipped: 66448 pix)
Append: sigclip.log ... OK
Write flatd_sky.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Dark image: flatd_sky.fits
Read: flat1_sky ... 
Read: mzrh00071336.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read flatd_sky.fits  

In [ ]:
!../sigclip sky sky.fits skyd.fits flat_sky.fits

Clipping threshold: 3 sigma
Dark image: skyd.fits
Flat image: flat_sky.fits
Read: sky ... 
Read: mzrh00071004.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read skyd.fits  ... 2160 x 1280 format Subtraction OK
Sigma  : 82.2803 ± 75.4501 (Clipped: 92434 pix)
Average: 110.448 ± 421.989 (Clipped: 461409 pix)
Append: sigclip.log ... OK
Read flat_sky.fits  ... 2160 x 1280 format Flat fielding OK
Write sky.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


In [ ]:
!../mzwarp sky.fits

Read: ../cal_H.dat ... OK
Read: sky.fits ... 2160 x 1280 format OK
Write: tmp.dat
tail -n+2 ../cal_H.dat >> tmp.dat && mv tmp.dat ../cal_H.dat
Y sigma = 0.0617809
Yave sigma = 0.000617594
Read: sky.fits ... 2160 x 1280 format OK
Number of detected lines: 43/43 (offset -37)
X sigma = -0.509396
Write: mzwarp.dat ... OK
gnuplot < ../mzwarp.plt
sh: 1: cannot open ../mzwarp.plt: No such file
Xave sigma = 0.000364196
Anchor0: 651.253386 91.672605
Anchor1: 653.859681 633.588272
Anchor2: 4016.911127 90.091567
Anchor3: 4019.923247 631.680752
Write mzwarp.fits ... 4848 x 720 format OK
Write mzwarp_dxdy.fits ... 4848 x 720 x 2 format OK


In [ ]:
!../double sky.fits SKY.fits mzwarp_dxdy.fits

Read: ../mzr_bundle.pos ... OK
Read: ../cal_H.dat ... OK
Read: flat.dat ... Can not open.
Relative flux correction will not be applied.
Read: flats.fits ... Can not open.
Relative spectum correction will not be applied.
Read: sky.fits ... Read: mzwarp_dxdy.fits ... icen=628 => 628 4848 x 720 x 2 format OK
Read: sky.fits ... 2160 x 1280 format gain=x16 OK
av:967.148 sg:2069.05
Write: double.bmp ... OK
Fiber flux (min/ave/max/bave): 8258.08/9602/10168.1/6491.42
Sky level: 0
Fiber flux (min/ave/max/bave): 8256.24/9598.94/10164.3/6489.59
Sky level: 0
Fiber flux (min/ave/max/bave): 8256.25/9598.94/10164.3/6489.59
Sky level: 0
Kernel:  0.989726 0.994606 0.99802 0.999565 1 0.999558 0.99802 0.99467 0.989772
Fiber flux (min/ave/max/bave): 8256.25/9598.94/10164.3/6489.59
Write double_im.fits ... 40 x 40 format OK
Write: double_y.dat ... OK
Center : -0.0165055 -0.00498036
Sigma :  3.31612
Fave  : 9598.94
S/N    : 217.263
Write: double_x.dat ... F_cont=9997.5-61.0638*(w-654.1), EW_line=-0.440573 /

In [ ]:
!cp double_rf.fits flats.fits
!cp double_y.dat flat.dat
!../double sky.fits SKY.fits mzwarp_dxdy.fits
!cp double_rf.fits flats.fits
!cp double_y.dat flat.dat
!../double sky.fits SKY.fits mzwarp_dxdy.fits
!cp double_rf.fits flats.fits
!cp double_y.dat flat.dat
!../double sky.fits SKY.fits mzwarp_dxdy.fits

Read: ../mzr_bundle.pos ... OK
Read: ../cal_H.dat ... OK
Read: flat.dat ... OK
Read: flats.fits ... 4848 x 61 format OK
Read: sky.fits ... Read: mzwarp_dxdy.fits ... icen=628 => 628 4848 x 720 x 2 format OK
Read: sky.fits ... 2160 x 1280 format gain=x16 OK
av:967.148 sg:2069.05
Write: double.bmp ... OK
Fiber flux (min/ave/max/bave): 0.967337/4003.16/9998.53/2707.12
Sky level: 0.031348
Fiber flux (min/ave/max/bave): 0.967018/4001.85/9994.99/2706.52
Sky level: 0.0313485
Fiber flux (min/ave/max/bave): 0.96702/4001.85/9995.01/2706.61
Sky level: 0.0313485
Kernel:  0.98972 0.994607 0.99802 0.999563 1 0.999557 0.998019 0.994672 0.989783
Fiber flux (min/ave/max/bave): 0.96702/4001.85/9995.01/2706.61
Write double_im.fits ... 40 x 40 format OK
Write: double_y.dat ... OK
Center : -0.354277 -0.01224
Sigma :  3.16008
Fave  : 9599.15
S/N    : 217.253
Write: double_x.dat ... F_cont=9997.78-61.0602*(w-654.1), EW_line=-0.44063 / -0.152929 OK
cp double_x.dat SKY_x.dat
Write: double_im.bmp ... OK
Write d

Check Anchor files

In [ ]:
!grep ANCHOR mzrh.log | cat -n
!grep ANCHOR mzrh.log > anc

     1	mzrh00069764.fits 2026-09-21 18:03:48.891859    1.00 x01 -20 -21.9  37.3  24.9  25.2  24.9  24.3   896.14  702.86   646 65535 18086 ANCHOR Shut FeNe Fan
     2	mzrh00069765.fits 2026-09-21 18:03:49.940034    1.00 x01 -20 -21.9  37.1  24.8  25.2  24.9  24.3   895.18  702.53   654 65535 18086 ANCHOR FeNe Fan
     3	mzrh00069766.fits 2026-09-21 18:03:50.980098    1.00 x01 -20 -21.9  37.1  24.8  25.2  24.8  24.3   895.36  702.47   667 65535 18086 ANCHOR FeNe Fan
     4	mzrh00069767.fits 2026-09-21 18:03:52.020113    1.00 x01 -20 -21.9  37.1  24.8  25.2  24.8  24.3   896.98  702.59   694 65535 18086 ANCHOR FeNe Fan
     5	mzrh00069768.fits 2026-09-21 18:03:53.060178    1.00 x01 -20 -21.9  37.1  24.9  25.2  24.8  24.4   896.76  702.65   674 65535 18086 ANCHOR FeNe Fan
     6	mzrh00069769.fits 2026-09-21 18:03:54.100037    1.00 x01 -20 -21.9  37.1  24.9  25.2  24.8  24.3   897.17  702.58   670 65535 18086 ANCHOR FeNe Fan
     7	mzrh00069770.fits 2026-09-21 18:03:55.140063    1.00 x01 -

In [ ]:
!grep DARK mzrh.log | grep FeNe | cat -n

     1	mzrh00069754.fits 2026-09-21 18:03:25.975393    1.00 x01 -20 -22.0  37.4  24.8  25.1  24.9  24.4   882.04   45.58   678  1204 18086 DARK FeNe Fan
     2	mzrh00069755.fits 2026-09-21 18:03:27.026987    1.00 x01 -20 -21.9  37.3  24.8  25.0  24.9  24.4   882.64   42.88   679  1195 18086 DARK Shut FeNe Fan
     3	mzrh00069756.fits 2026-09-21 18:03:28.067060    1.00 x01 -20 -21.9  37.3  24.8  25.0  24.8  24.4   879.60   44.75   657  1185 18086 DARK Shut FeNe Fan
     4	mzrh00069757.fits 2026-09-21 18:03:29.106931    1.00 x01 -20 -21.9  37.3  24.8  25.0  24.8  24.4   882.12   45.29   672  1232 18086 DARK Shut FeNe Fan
     5	mzrh00069758.fits 2026-09-21 18:03:30.146894    1.00 x01 -20 -21.9  37.3  24.7  25.1  24.8  24.4   881.45   44.85   676  3778 18086 DARK Shut FeNe Fan
     6	mzrh00069759.fits 2026-09-21 18:03:31.186805    1.00 x01 -20 -21.9  37.3  24.8  25.1  24.9  24.4   881.89   44.13   665  1158 18086 DARK Shut FeNe Fan
     7	mzrh00069760.fits 2026-09-21 18:03:32.226645    1.

In [ ]:
!grep DARK mzrh.log | grep FeNe > ancd

In [ ]:
!split -d -l 10 -a 3 anc anc_
!split -d -l 10 -a 3 ancd ancd_

In [ ]:
!ls anc_??? > tmp1
!ls ancd_??? > tmp2
!paste -d " " tmp2 tmp2 | sed -e 's/^/..\/sigclip /g' | sed -e 's/$/.fits/g' > tmp3
!cat tmp3
!sh tmp3
!paste -d " " tmp1 tmp1 | sed -e 's/^/..\/sigclip /g' | sed -e 's/$/.fits/g' > tmp3
!ls ancd_*.fits > tmp4
!paste -d" " tmp3 tmp4 | sed -e 's/$/ flat.fits/g' > tmp5
!cat tmp5
!sh tmp5

../sigclip ancd_000 ancd_000.fits
../sigclip ancd_001 ancd_001.fits
../sigclip ancd_002 ancd_002.fits
../sigclip ancd_003 ancd_003.fits
../sigclip ancd_004 ancd_004.fits
../sigclip ancd_005 ancd_005.fits
../sigclip ancd_006 ancd_006.fits
../sigclip ancd_007 ancd_007.fits
../sigclip ancd_008 ancd_008.fits
../sigclip ancd_009 ancd_009.fits
../sigclip ancd_010 ancd_010.fits
Clipping threshold: 3 sigma
Read: ancd_000 ... 
Read: mzrh00069763.fits ... OK
Maximum rejected pixel: [2160,1280] 0/10 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 18.4733 ± 4.67029 (Clipped: 16723 pix)
Average: 881.146 ± 38.4482 (Clipped: 18512 pix)
Append: sigclip.log ... OK
Write ancd_000.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Read: ancd_001 ... 
Read: mzrh00070183.fits ... OK
Maximum rejected pixel: [2160,1280] 0/10 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 18.4887 ± 4.67598 (Clipped: 14131 pix)
Average: 878.05 ± 38

In [ ]:
!cat mzrh.log

mzrh00069742.fits 2026-09-21 18:02:01.324496    1.00 x01 -20 -21.8  37.0  24.6  24.8  24.9  24.4   879.42   45.69   670  1246 18150 FOCUSING Shut FeNe Fan
mzrh00069743.fits 2026-09-21 18:02:37.450391    1.00 x01 -20 -22.0  37.6  24.6  24.9  24.8  24.4   893.30  714.09   665 65535 17600 FOCUSING FeNe Fan
mzrh00069744.fits 2026-09-21 18:02:40.609605    1.00 x01 -20 -22.0  37.5  24.6  24.9  24.9  24.4   894.24  710.12   680 65535 17700 FOCUSING FeNe Fan
mzrh00069745.fits 2026-09-21 18:02:43.908356    1.00 x01 -20 -22.0  37.5  24.6  24.9  24.8  24.4   893.27  705.48   674 65535 17800 FOCUSING FeNe Fan
mzrh00069746.fits 2026-09-21 18:02:47.188317    1.00 x01 -20 -22.0  37.5  24.6  24.9  24.8  24.4   895.01  702.57   671 65535 17900 FOCUSING FeNe Fan
mzrh00069747.fits 2026-09-21 18:02:50.492691    1.00 x01 -20 -22.0  37.5  24.7  24.9  24.9  24.4   897.98  701.62   660 65535 18000 FOCUSING FeNe Fan
mzrh00069748.fits 2026-09-21 18:02:53.945688    1.00 x01 -20 -22.0  37.5  24.7  25.0  24.8  24.

In [ ]:
target = "EV_Lac"
!grep {target} mzrh.log
!grep {target} mzrh.log > {target}_H

mzrh00070738.fits 2026-09-21 23:45:44.103065   60.00 x16 -20 -20.0  28.8  22.5  23.1  21.1  21.0  1406.54  367.21   493 65535 18042 EV_Lac Fan
mzrh00070739.fits 2026-09-21 23:46:44.368778   60.00 x16 -20 -20.0  28.8  22.5  23.1  21.1  21.0  1405.31  346.22   518 65535 18042 EV_Lac Fan
mzrh00070740.fits 2026-09-21 23:47:44.653146   60.00 x16 -20 -20.0  28.8  22.5  23.1  21.1  21.0  1401.82  332.82   596 65535 18042 EV_Lac Fan
mzrh00070741.fits 2026-09-21 23:48:44.918479   60.00 x16 -20 -20.0  28.8  22.5  23.1  21.1  21.0  1404.44  339.22   527 65535 18042 EV_Lac Fan
mzrh00070742.fits 2026-09-21 23:49:45.183953   60.00 x16 -20 -20.0  28.8  22.5  23.1  21.1  21.0  1407.94  350.51   739 65535 18042 EV_Lac Fan


Reduce Target files

In [ ]:
!../reduc {target}_H {target}_rH bias.fits flat.fits

Read: ../cal_H.dat ... OK
icen=661
Read bias.fits ... 2160 x 1280 format OK
Read ../ampgH1.fits ... 2160 x 1280 format OK
Read ../ampgH2.fits ... 2160 x 1280 format OK
Read ../ampgH3.fits ... 2160 x 1280 format OK
Read flat.fits  ... 2160 x 1280 format icen=626 OK
Read: EV_Lac_H ... 
Read: EV_Lac_H ... 
Read: mzrh00070738.fits ... gain=x16 (1) OK
N_CR=25  R_CRmax=6
Ampglow correction factor: -0.122323*ampgH1.fits+0.192087*ampgH2.fits-0.0352158*ampgH3.fits-12.741
Bias correction
Write EV_Lac_rH_001.fits ... 2160 x 1280 format OK
Read: mzrh00070739.fits ... gain=x16 (1) OK
N_CR=18  R_CRmax=3
Ampglow correction factor: -0.123442*ampgH1.fits+0.193733*ampgH2.fits-0.0353772*ampgH3.fits-14.5025
Bias correction
Write EV_Lac_rH_002.fits ... 2160 x 1280 format OK
Read: mzrh00070740.fits ... gain=x16 (1) OK
N_CR=21  R_CRmax=3
Ampglow correction factor: -0.120075*ampgH1.fits+0.18971*ampgH2.fits-0.0352381*ampgH3.fits-14.0732
Bias correction
Write EV_Lac_rH_003.fits ... 2160 x 1280 format OK
Read: m

In [ ]:
!rm spec_*.png {target}_H.log
!../double {target}_rH {target}_dH mzwarp_dxdy.fits anc

rm: cannot remove 'spec_*.png': No such file or directory
rm: cannot remove 'EV_Lac_H.log': No such file or directory
Read: ../mzr_bundle.pos ... OK
Read: ../cal_H.dat ... OK
Read: flat.dat ... OK
Read: flats.fits ... 4848 x 61 format OK
Read: EV_Lac_rH ... Read: mzwarp_dxdy.fits ... icen=661 => 628 4848 x 720 x 2 format OK
*** Anchor correction ***
OK
Peak [651,89]: 3561.8 dr/dx/dy=2.402/2.015/2.016 (650.342 87.140) @23:42:57.872006
Peak [653,629]: 6084.21 dr/dx/dy=2.421/2.060/2.022 (652.751 628.800) @23:42:57.872006
Peak [4015,85]: 2536.22 dr/dx/dy=2.384/1.999/2.009 (4014.958 84.290) @23:42:57.872006
Peak [4017,627]: 3243.88 dr/dx/dy=2.422/2.043/2.043 (4016.858 626.813) @23:42:57.872006
icen=628 => 627
(650.342 89.140) => (650.253 90.673)
(652.751 630.800) => (652.860 632.588)
(4014.958 86.290) => (4015.911 89.092)
(4016.858 628.813) => (4018.923 630.681)
EV_Lac_rH_001.fits ... 2160 x 1280 format gain=x16 OK
av:76.772 sg:290.473
Write: double.bmp ... OK
Fiber flux (min/ave/max/bave):

In [ ]:
!convert -delay 50 -loop 0 spec_*.png {target}_H.gif
!rm spec_*.png
!../merge {target}_dH
!convert -delay 50 -loop 0 spec_*.png {target}_rH.gif

/bin/bash: line 1: convert: command not found
rm: cannot remove 'spec_*.png': No such file or directory
Read: EV_Lac_dH_001_x.dat ... 0.038970 0.081539 2.054210 709.729980 OK
Read: EV_Lac_dH_002_x.dat ... 0.032544 0.094396 2.063700 711.085022 OK
Read: EV_Lac_dH_003_x.dat ... 0.004439 0.014138 2.206630 699.518005 OK
Read: EV_Lac_dH_004_x.dat ... 0.025321 0.137280 2.113660 702.659973 OK
Read: EV_Lac_dH_005_x.dat ... 0.039503 0.116546 2.036170 712.093994 OK
Loop ends. Number of files: 5
ndat=5
Write: EV_Lac_dH_x.dat ... OK
Write: EV_Lac_dH_001_x.dat ... OK
sed -e 's/FILENAME/EV\\_Lac\\_dH\\_001\\_x.dat/g' ../merge.plt.template | sed -e 's/FNAME/EV_Lac_dH_001_x.dat/g' > merge.plt
sed: can't read ../merge.plt.template: No such file or directory
gnuplot < merge.plt
sh: 1: gnuplot: not found
Write: EV_Lac_dH_002_x.dat ... OK
sed -e 's/FILENAME/EV\\_Lac\\_dH\\_002\\_x.dat/g' ../merge.plt.template | sed -e 's/FNAME/EV_Lac_dH_002_x.dat/g' > merge.plt
sed: can't read ../merge.plt.template: No suc

In [ ]:
!../frcor {target}_dH_x.dat > {target}_dH_xf.dat

Move to Ca directory

In [ ]:
%cd ../
%cd ./{date}C

/content/drive/MyDrive/MIDSSAR/20260921
/content/drive/MyDrive/MIDSSAR/20260921/20260921C


Check observation log (Ca)

In [ ]:
!cat mzrc.log

mzrc00067401.fits 2026-09-22 15:58:47.172731    2.00 x16 -20 -17.1  35.2  26.7  26.9  26.3  26.0  1229.40   85.16   619 17381 18150 FOCUSING Shut FeNe Fan
mzrc00067402.fits 2026-09-22 15:59:23.212850    2.00 x16 -20 -17.5  35.5  26.8  26.9  26.3  26.0  6313.88 12171.09   678 65535 17600 FOCUSING FeNe Fan
mzrc00067403.fits 2026-09-22 15:59:26.605958    2.00 x16 -20 -19.4  36.9  26.8  26.9  26.3  26.0  6292.07 12319.18   664 65535 17700 FOCUSING FeNe Fan
mzrc00067404.fits 2026-09-22 15:59:29.982029    2.00 x16 -20 -19.6  37.0  26.8  26.9  26.3  26.0  6269.39 12469.65   616 65535 17800 FOCUSING FeNe Fan
mzrc00067405.fits 2026-09-22 15:59:33.257571    2.00 x16 -20 -19.7  37.1  26.8  26.9  26.3  26.0  6243.62 12625.76   667 65535 17900 FOCUSING FeNe Fan
mzrc00067406.fits 2026-09-22 15:59:36.532669    2.00 x16 -20 -19.8  37.2  26.8  26.9  26.3  26.0  6231.48 12738.22   630 65535 18000 FOCUSING FeNe Fan
mzrc00067407.fits 2026-09-22 15:59:39.895957    2.00 x16 -20 -20.0  37.3  26.8  26.9  26.3

Check Bias files (Ca)

In [ ]:
!grep BIAS mzrc.log | cat -n
!grep BIAS mzrc.log  > bias

     1	mzrc00066599.fits 2026-09-21 18:04:22.084122    0.01 x16 -20 -21.7  33.5  24.9  25.2  24.8  24.3  1181.11   77.95   552  2962 18086 BIAS Shut Fan
     2	mzrc00066600.fits 2026-09-21 18:04:22.140326    0.01 x16 -20 -21.7  33.3  24.9  25.2  24.8  24.3  1180.19   79.34   661  3080 18086 BIAS Shut Fan
     3	mzrc00066601.fits 2026-09-21 18:04:22.188999    0.01 x16 -20 -21.7  33.3  24.9  25.2  24.8  24.3  1179.03   79.40   417  3137 18086 BIAS Shut Fan
     4	mzrc00066602.fits 2026-09-21 18:04:22.237172    0.01 x16 -20 -21.7  33.3  24.9  25.2  24.8  24.3  1179.52   78.86   539  3048 18086 BIAS Shut Fan
     5	mzrc00066603.fits 2026-09-21 18:04:22.289228    0.01 x16 -20 -21.7  33.3  24.9  25.2  24.8  24.3  1179.71   79.67   562  2971 18086 BIAS Shut Fan
     6	mzrc00066604.fits 2026-09-21 18:04:22.337336    0.01 x16 -20 -21.7  33.3  24.9  25.2  24.8  24.3  1180.65   78.96   586  3058 18086 BIAS Shut Fan
     7	mzrc00066605.fits 2026-09-21 18:04:22.385285    0.01 x16 -20 -21.7  33.3  2

Make master Bias (Ca)

In [ ]:
!../sigclip bias bias.fits

Clipping threshold: 3 sigma
Read: bias ... 
Read: mzrc00066698.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 38.0796 ± 5.40251 (Clipped: 112870 pix)
Average: 1174.72 ± 52.5271 (Clipped: 57947 pix)
Append: sigclip.log ... OK
Write bias.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


Check Flat files (Ca)

In [ ]:
!grep INSTFLAT mzrc.log | cat -n

     1	mzrc00066799.fits 2026-09-21 18:30:11.908932    0.02 x01 -20 -20.0  29.3  25.0  25.0  24.2  23.2  2090.77 3011.65   671 16778 14086 INSTFLAT Cont Fan
     2	mzrc00066800.fits 2026-09-21 18:30:11.975099    0.02 x01 -20 -19.9  29.3  25.0  24.9  24.1  23.2  2089.93 3012.56   664 16748 14086 INSTFLAT Cont Fan
     3	mzrc00066801.fits 2026-09-21 18:30:12.032784    0.02 x01 -20 -19.9  29.3  25.0  24.9  24.1  23.2  2092.29 3013.69   658 16885 14086 INSTFLAT Cont Fan
     4	mzrc00066802.fits 2026-09-21 18:30:12.090321    0.02 x01 -20 -19.9  29.3  25.0  24.9  24.1  23.2  2089.14 3013.95   668 16805 14086 INSTFLAT Cont Fan
     5	mzrc00066803.fits 2026-09-21 18:30:12.148126    0.02 x01 -20 -19.9  29.3  25.0  24.9  24.1  23.2  2089.58 3013.99   674 16733 14086 INSTFLAT Cont Fan
     6	mzrc00066804.fits 2026-09-21 18:30:12.205553    0.02 x01 -20 -19.9  29.3  25.0  24.9  24.1  23.2  2090.07 3014.44   675 16711 14086 INSTFLAT Cont Fan
     7	mzrc00066805.fits 2026-09-21 18:30:12.263286    0.0

In [ ]:
!grep INSTFLAT mzrc.log | sed -n '001,100p' > flat1
!grep INSTFLAT mzrc.log | sed -n '101,200p' > flat2
#!grep INSTFLAT mzrc.log | sed -n '201,300p' >> flat1
#!grep INSTFLAT mzrc.log | sed -n '301,400p' >> flat2
#!grep INSTFLAT mzrc.log | sed -n '401,500p' >> flat1
#!grep INSTFLAT mzrc.log | sed -n '501,600p' >> flat2

In [ ]:
!grep DARK mzrc.log | grep Cont | cat -n
!grep DARK mzrc.log | grep Cont > flatd

     1	mzrc00066699.fits 2026-09-21 18:30:00.311227    0.02 x01 -20 -20.1  29.3  25.0  24.9  24.1  23.2   851.27   43.22   663  1122 18086 DARK Shut Cont Fan
     2	mzrc00066700.fits 2026-09-21 18:30:00.375294    0.02 x01 -20 -20.0  29.3  25.0  24.9  24.1  23.2   852.91   42.33   632  1119 18086 DARK Shut Cont Fan
     3	mzrc00066701.fits 2026-09-21 18:30:00.433190    0.02 x01 -20 -20.0  29.3  25.0  24.9  24.1  23.2   852.64   41.30   669  1111 18086 DARK Shut Cont Fan
     4	mzrc00066702.fits 2026-09-21 18:30:00.495396    0.02 x01 -20 -20.0  29.3  25.0  24.9  24.1  23.2   852.30   43.77   645  1124 18086 DARK Shut Cont Fan
     5	mzrc00066703.fits 2026-09-21 18:30:00.557834    0.02 x01 -20 -20.0  29.3  25.0  24.9  24.1  23.2   851.36   42.88   652  1125 18086 DARK Shut Cont Fan
     6	mzrc00066704.fits 2026-09-21 18:30:00.620822    0.02 x01 -20 -20.0  29.3  25.0  24.9  24.1  23.2   853.02   42.03   653  1139 18086 DARK Shut Cont Fan
     7	mzrc00066705.fits 2026-09-21 18:30:00.683250 

Make master Flat (Ca)

In [ ]:
!../sigclip flatd flatd.fits

Clipping threshold: 3 sigma
Read: flatd ... 
Read: mzrc00066798.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 19.8277 ± 1.58573 (Clipped: 5586 pix)
Average: 847.755 ± 30.7112 (Clipped: 89184 pix)
Append: sigclip.log ... OK
Write flatd.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


In [ ]:
!../sigclip flat1 flat1.fits flatd.fits
!../sigclip flat2 flat2.fits flatd.fits

Clipping threshold: 3 sigma
Dark image: flatd.fits
Read: flat1 ... 
Read: mzrc00066898.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read flatd.fits  ... 2160 x 1280 format Subtraction OK
Bright : [1:2160,478:732]
Center : [1:2161,505:705]
Sigma  : 74.4369 ± 23.1975 (Clipped: 2 pix)
Average: 6582.46 ± 3882.71 (Clipped: 0 pix)
Append: sigclip.log ... OK
Normalization OK
Write flat1.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Dark image: flatd.fits
Read: flat2 ... 
Read: mzrc00066998.fits ... OK
Maximum rejected pixel: [2160,1280] 1/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read flatd.fits  ... 2160 x 1280 format Subtraction OK
Bright : [1:2160,476:733]
Center : [1:2161,504:704]
Sigma  : 75.1137 ± 23.2133 (Clipped: 6 pix)
Average: 6732.59 ± 3944.98 (Clipped: 0 pix)
Append: sigclip.log ... OK
Normalization OK
Write flat2.fits ... 2160 x 1280 format OK
Write

In [ ]:
!../ave flat1.fits flat2.fits flat0.fits
!../mkflat flat0
!mv flat0_xy.fits flat.fits

Read: flat1.fits ... 2160 x 1280 format OK
Read: flat2.fits ... 2160 x 1280 format OK
Write flat0.fits ... 2160 x 1280 format OK
Read: flat0.fits ... 2160 x 1280 format OK
Bright : [53:1621,487:724]
Center : [1:1837,505:705]
 1.067551 0.933574 1.057551 0.936057
 0.965757 1.052566 0.944326 1.072620
 1.057623 0.939177 1.054873 0.938233
 0.939859 1.050004 0.937485 1.052744
Write flat0_xy.fits ... 2160 x 1280 format OK
Write flat0_res.fits ... 2160 x 1280 format OK


Check Skyflat files (Ca)

In [ ]:
!pwd

/content/drive/MyDrive/MIDSSAR/20260921/20260921C


In [ ]:
!cp -r /content/drive/MyDrive/MIDSSAR/data/20260922/20260922C/{mzwarp_dxdy.fits,flats.fits,flat.dat} /content/drive/MyDrive/MIDSSAR/20260921/20260921C/

In [ ]:
!grep SKYFLAT mzrc.log | cat -n

     1	mzrc00067453.fits 2026-09-22 16:02:24.270262    1.00 x16 -20 -22.6  39.3  27.1  27.5  26.3  26.0  3685.12 6424.74   629 49658 18046 SKYFLAT Fan Cl2
     2	mzrc00067454.fits 2026-09-22 16:02:25.314584    1.00 x16 -20 -22.4  39.2  27.2  27.5  26.3  26.0  3683.93 6428.30   597 49556 18046 SKYFLAT Fan Cl2
     3	mzrc00067455.fits 2026-09-22 16:02:26.352636    1.00 x16 -20 -22.4  39.2  27.2  27.5  26.3  26.0  3682.51 6426.92   653 49380 18046 SKYFLAT Fan Cl2
     4	mzrc00067456.fits 2026-09-22 16:02:27.390605    1.00 x16 -20 -22.4  39.2  27.2  27.5  26.3  26.0  3683.30 6423.99   589 49352 18046 SKYFLAT Fan Cl2
     5	mzrc00067457.fits 2026-09-22 16:02:28.428452    1.00 x16 -20 -22.4  39.2  27.2  27.5  26.3  26.0  3684.45 6419.29   681 49257 18046 SKYFLAT Fan Cl2
     6	mzrc00067458.fits 2026-09-22 16:02:29.466374    1.00 x16 -20 -22.4  39.2  27.2  27.6  26.3  26.0  3684.54 6428.98   638 49439 18046 SKYFLAT Fan Cl2
     7	mzrc00067459.fits 2026-09-22 16:02:30.504385    1.00 x16 -20 -2

In [ ]:
!grep SKYFLAT mzrc.log | sed -n '101,200p' > sky
!grep DARK mzrc.log | grep -v Cont | grep -v FeNe | cat -n

     1	mzrc00067443.fits 2026-09-22 16:02:11.316101    1.00 x16 -20 -22.6  39.1  27.1  27.5  26.3  26.0  1209.83   89.63   567 39400 18046 DARK Shut Fan Cl2
     2	mzrc00067444.fits 2026-09-22 16:02:12.369849    1.00 x16 -20 -22.6  39.3  27.1  27.5  26.3  26.0  1209.27   86.38   523  6466 18046 DARK Shut Fan Cl2
     3	mzrc00067445.fits 2026-09-22 16:02:13.408155    1.00 x16 -20 -22.6  39.3  27.1  27.5  26.3  26.0  1212.31   86.16   513  4180 18046 DARK Shut Fan Cl2
     4	mzrc00067446.fits 2026-09-22 16:02:14.446119    1.00 x16 -20 -22.6  39.3  27.1  27.5  26.3  26.0  1211.54   86.09   565 23209 18046 DARK Shut Fan Cl2
     5	mzrc00067447.fits 2026-09-22 16:02:15.484162    1.00 x16 -20 -22.6  39.3  27.1  27.5  26.3  26.0  1212.16   85.97   605  4774 18046 DARK Shut Fan Cl2
     6	mzrc00067448.fits 2026-09-22 16:02:16.522090    1.00 x16 -20 -22.6  39.3  27.1  27.5  26.3  26.0  1212.94   86.18   592 10706 18046 DARK Shut Fan Cl2
     7	mzrc00067449.fits 2026-09-22 16:02:17.560290    1.0

In [ ]:
!grep DARK mzrc.log | grep -v Cont | grep -v FeNe | sed -n '11,20p' > skyd

Make master Skyflat

In [ ]:
!../sigclip skyd skyd.fits

Clipping threshold: 3 sigma
Read: skyd ... 
Read: mzrc00067562.fits ... OK
Maximum rejected pixel: [2160,1280] 0/10 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 37.5527 ± 10.5389 (Clipped: 42284 pix)
Average: 1199.06 ± 57.9639 (Clipped: 54278 pix)
Append: sigclip.log ... OK
Write skyd.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


In [ ]:
!grep BIAS mzrc.log

mzrc00067695.fits 2026-09-22 18:36:45.588917    0.01 x16 -20 -21.9  32.3  23.7  23.8  24.2  23.6  1178.53   77.97   552  2963 17944 BIAS Shut Fan
mzrc00067696.fits 2026-09-22 18:36:45.643391    0.01 x16 -20 -21.8  32.2  23.7  23.8  24.2  23.6  1177.27   78.41   520  2998 17944 BIAS Shut Fan
mzrc00067697.fits 2026-09-22 18:36:45.691785    0.01 x16 -20 -21.8  32.2  23.7  23.8  24.2  23.6  1178.40   79.43   514  2963 17944 BIAS Shut Fan
mzrc00067698.fits 2026-09-22 18:36:45.739909    0.01 x16 -20 -21.8  32.2  23.7  23.8  24.2  23.6  1179.18   79.35   549  3002 17944 BIAS Shut Fan
mzrc00067699.fits 2026-09-22 18:36:45.788384    0.01 x16 -20 -21.8  32.2  23.7  23.8  24.2  23.6  1178.55   79.14   619  3229 17944 BIAS Shut Fan
mzrc00067700.fits 2026-09-22 18:36:45.838845    0.01 x16 -20 -21.8  32.2  23.7  23.8  24.2  23.6  1180.29   78.13   612  3124 17944 BIAS Shut Fan
mzrc00067701.fits 2026-09-22 18:36:45.890728    0.01 x16 -20 -21.8  32.2  23.7  23.8  24.2  23.6  1178.49   79.35   610  303

In [ ]:
!grep BIAS mzrc.log | sed -n '1,100p' > bias_sky
!../sigclip bias_sky bias_sky.fits
!grep INSTFLAT mzrc.log | sed -n '001,100p' > flat1_sky
!grep INSTFLAT mzrc.log | sed -n '101,200p' > flat2_sky
#!grep INSTFLAT mzrc.log | sed -n '201,300p' >> flat1_sky
#!grep INSTFLAT mzrc.log | sed -n '301,400p' >> flat2_sky
!grep DARK mzrc.log | grep Cont | sed -n '1,200p'> flatd_sky
!../sigclip flatd_sky flatd_sky.fits
!../sigclip flat1_sky flat1_sky.fits flatd_sky.fits
!../sigclip flat2_sky flat2_sky.fits flatd_sky.fits
!../ave flat1_sky.fits flat2_sky.fits flat0_sky.fits
!../mkflat flat0_sky
!mv flat0_sky_xy.fits flat_sky.fits

Clipping threshold: 3 sigma
Read: bias_sky ... 
Read: mzrc00067794.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 38.0507 ± 5.40066 (Clipped: 112723 pix)
Average: 1173.43 ± 52.7045 (Clipped: 57642 pix)
Append: sigclip.log ... OK
Write bias_sky.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Read: flatd_sky ... 
Read: mzrc00067894.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 19.788 ± 1.591 (Clipped: 5996 pix)
Average: 846.488 ± 28.8437 (Clipped: 126163 pix)
Append: sigclip.log ... OK
Write flatd_sky.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Dark image: flatd_sky.fits
Read: flat1_sky ... 
Read: mzrc00067994.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Read flatd_sky.fits  ... 

In [ ]:
!../sigclip sky sky.fits skyd.fits flat_sky.fits

Clipping threshold: 3 sigma
Dark image: skyd.fits
Flat image: flat_sky.fits
Read: sky ... 
Read: mzrc00067662.fits ... OK
Maximum rejected pixel: [2160,1280] 0/100 rejected
Saturated (>64000 ADU) pixels: 0.01 pix/frame
Read skyd.fits  ... 2160 x 1280 format Subtraction OK
Sigma  : 47.3042 ± 9.36058 (Clipped: 495298 pix)
Average: 29.4111 ± 100.139 (Clipped: 494534 pix)
Append: sigclip.log ... OK
Read flat_sky.fits  ... 2160 x 1280 format Flat fielding OK
Write sky.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK


In [ ]:
!../mzwarp sky.fits

Read: ../cal_C.dat ... OK
Read: sky.fits ... 2160 x 1280 format OK
Write: tmp.dat
tail -n+2 ../cal_C.dat >> tmp.dat && mv tmp.dat ../cal_C.dat
Y sigma = 0.0580969
Yave sigma = 0.00750455
Read: sky.fits ... 2160 x 1280 format OK
Number of detected lines: 63/66 (offset 36)
X sigma = 0.55145
Write: mzwarp.dat ... OK
gnuplot < ../mzwarp.plt
sh: 1: cannot open ../mzwarp.plt: No such file
Xave sigma = 0.000520327
Anchor0: 886.631462 120.954974
Anchor1: 886.942536 604.540841
Anchor2: 4254.108491 121.330320
Anchor3: 4254.230747 603.231791
Write mzwarp.fits ... 4848 x 720 format OK
Write mzwarp_dxdy.fits ... 4848 x 720 x 2 format OK


In [ ]:
!../double sky.fits SKY.fits mzwarp_dxdy.fits

Read: ../mzr_bundle.pos ... OK
Read: ../cal_C.dat ... OK
Read: flat.dat ... Can not open.
Relative flux correction will not be applied.
Read: flats.fits ... Can not open.
Relative spectum correction will not be applied.
Read: sky.fits ... Read: mzwarp_dxdy.fits ... icen=605 => 605 4848 x 720 x 2 format OK
Read: sky.fits ... 2160 x 1280 format gain=x16 OK
av:1248.33 sg:3271.31
Write: double.bmp ... OK
Fiber flux (min/ave/max/bave): 8678.18/11467.5/12599.1/7374.78
Sky level: 0
Fiber flux (min/ave/max/bave): 8676.6/11465.6/12596.8/7373.8
Sky level: 0
Fiber flux (min/ave/max/bave): 8676.6/11465.6/12596.8/7373.8
Sky level: 0
Kernel:  0.985657 0.992765 0.997478 0.999512 1 0.999509 0.997476 0.992693 0.985637
Fiber flux (min/ave/max/bave): 8676.6/11465.6/12596.8/7373.8
Write double_im.fits ... 40 x 40 format OK
Write: double_y.dat ... OK
Center : -0.0262226 -0.000742119
Sigma :  3.32827
Fave  : 11465.6
S/N    : 242.826
Write: double_x.dat ... F_cont=13416.9+635.992*(w-394.425), EW_line=-1.9894

In [ ]:
!cp double_rf.fits flats.fits
!cp double_y.dat flat.dat
!../double sky.fits SKY.fits mzwarp_dxdy.fits
!cp double_rf.fits flats.fits
!cp double_y.dat flat.dat
!../double sky.fits SKY.fits mzwarp_dxdy.fits
!cp double_rf.fits flats.fits
!cp double_y.dat flat.dat
!../double sky.fits SKY.fits mzwarp_dxdy.fits

Read: ../mzr_bundle.pos ... OK
Read: ../cal_C.dat ... OK
Read: flat.dat ... OK
Read: flats.fits ... 4848 x 61 format OK
Read: sky.fits ... Read: mzwarp_dxdy.fits ... icen=605 => 605 4848 x 720 x 2 format OK
Read: sky.fits ... 2160 x 1280 format gain=x16 OK
av:1248.33 sg:3271.31
Write: double.bmp ... OK
Fiber flux (min/ave/max/bave): 0.881493/4638.01/12628.8/2971.87
Sky level: 0.0279634
Fiber flux (min/ave/max/bave): 0.881339/4637.23/12626.6/2971.46
Sky level: 0.0279633
Fiber flux (min/ave/max/bave): 0.881339/4637.23/12626.6/2971.46
Sky level: 0.0279633
Kernel:  0.985663 0.992764 0.997478 0.999512 1 0.999509 0.997476 0.992696 0.985639
Fiber flux (min/ave/max/bave): 0.881339/4637.23/12626.6/2971.46
Write double_im.fits ... 40 x 40 format OK
Write: double_y.dat ... OK
Center : -0.381897 -0.00175363
Sigma :  3.16838
Fave  : 11466.4
S/N    : 242.807
Write: double_x.dat ... F_cont=13418+636.128*(w-394.425), EW_line=-1.98964 / -0.453484 OK
cp double_x.dat SKY_x.dat
Write: double_im.bmp ... OK

Check Anchor files (Ca)

In [ ]:
!grep ANCHOR mzrc.log | cat -n
!grep ANCHOR mzrc.log > anc

     1	mzrc00066589.fits 2026-09-21 18:03:48.748122    2.00 x16 -20 -21.8  33.6  24.8  25.2  24.9  24.4  1198.34  100.38   578 21368 18086 ANCHOR FeNe Fan
     2	mzrc00066590.fits 2026-09-21 18:03:50.793187    2.00 x16 -20 -21.8  33.6  24.8  25.2  24.9  24.3  1196.81   99.55   467 22414 18086 ANCHOR FeNe Fan
     3	mzrc00066591.fits 2026-09-21 18:03:52.838336    2.00 x16 -20 -21.8  33.6  24.8  25.2  24.8  24.4  1196.44   99.46   622 22792 18086 ANCHOR FeNe Fan
     4	mzrc00066592.fits 2026-09-21 18:03:54.883211    2.00 x16 -20 -21.8  33.6  24.9  25.2  24.8  24.3  1198.67  101.39   595 23498 18086 ANCHOR FeNe Fan
     5	mzrc00066593.fits 2026-09-21 18:03:56.928370    2.00 x16 -20 -21.8  33.6  24.8  25.2  24.8  24.3  1196.70   99.35   501 22207 18086 ANCHOR FeNe Fan
     6	mzrc00066594.fits 2026-09-21 18:03:58.973310    2.00 x16 -20 -21.8  33.5  24.8  25.2  24.8  24.3  1196.53  101.85   643 21951 18086 ANCHOR FeNe Fan
     7	mzrc00066595.fits 2026-09-21 18:04:01.018122    2.00 x16 -20 -2

In [ ]:
!grep DARK mzrc.log | grep FeNe | cat -n

     1	mzrc00066579.fits 2026-09-21 18:03:25.741243    2.00 x16 -20 -22.0  33.9  24.8  25.1  24.9  24.4  1197.37   84.56   562  9973 18086 DARK Shut FeNe Fan
     2	mzrc00066580.fits 2026-09-21 18:03:27.795664    2.00 x16 -20 -21.9  33.7  24.8  25.0  24.9  24.4  1197.08   86.05   547  7338 18086 DARK Shut FeNe Fan
     3	mzrc00066581.fits 2026-09-21 18:03:29.841311    2.00 x16 -20 -21.9  33.7  24.8  25.0  24.8  24.4  1195.33   85.76   500  7432 18086 DARK Shut FeNe Fan
     4	mzrc00066582.fits 2026-09-21 18:03:31.889798    2.00 x16 -20 -21.9  33.7  24.7  25.1  24.8  24.4  1196.17   85.94   534  7522 18086 DARK Shut FeNe Fan
     5	mzrc00066583.fits 2026-09-21 18:03:33.934110    2.00 x16 -20 -21.9  33.7  24.8  25.1  24.9  24.3  1196.80   85.90   516 12306 18086 DARK Shut FeNe Fan
     6	mzrc00066584.fits 2026-09-21 18:03:35.980705    2.00 x16 -20 -21.9  33.7  24.8  25.1  24.9  24.3  1196.86   84.72   591  7242 18086 DARK Shut FeNe Fan
     7	mzrc00066585.fits 2026-09-21 18:03:38.025201 

In [ ]:
!grep DARK mzrc.log | grep FeNe > ancd

In [ ]:
!split -d -l 10 -a 3 anc anc_
!split -d -l 10 -a 3 ancd ancd_

In [ ]:
!ls anc_??? > tmp1
!ls ancd_??? > tmp2
!paste -d " " tmp2 tmp2 | sed -e 's/^/..\/sigclip /g' | sed -e 's/$/.fits/g' > tmp3
!cat tmp3
!sh tmp3
!paste -d " " tmp1 tmp1 | sed -e 's/^/..\/sigclip /g' | sed -e 's/$/.fits/g' > tmp3
!ls ancd_*.fits > tmp4
!paste -d" " tmp3 tmp4 | sed -e 's/$/ flat.fits/g' > tmp5
!cat tmp5
!sh tmp5

../sigclip ancd_000 ancd_000.fits
../sigclip ancd_001 ancd_001.fits
../sigclip ancd_002 ancd_002.fits
../sigclip ancd_003 ancd_003.fits
../sigclip ancd_004 ancd_004.fits
../sigclip ancd_005 ancd_005.fits
../sigclip ancd_006 ancd_006.fits
../sigclip ancd_007 ancd_007.fits
../sigclip ancd_008 ancd_008.fits
../sigclip ancd_009 ancd_009.fits
../sigclip ancd_010 ancd_010.fits
Clipping threshold: 3 sigma
Read: ancd_000 ... 
Read: mzrc00066588.fits ... OK
Maximum rejected pixel: [2160,1280] 0/10 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 36.5578 ± 10.3385 (Clipped: 44667 pix)
Average: 1191.7 ± 59.0159 (Clipped: 52058 pix)
Append: sigclip.log ... OK
Write ancd_000.fits ... 2160 x 1280 format OK
Write sigclip.fits ... 2160 x 1280 format OK
Clipping threshold: 3 sigma
Read: ancd_001 ... 
Read: mzrc00067008.fits ... OK
Maximum rejected pixel: [2160,1280] 0/10 rejected
Saturated (>64000 ADU) pixels: 0 pix/frame
Sigma  : 36.6761 ± 10.3613 (Clipped: 44869 pix)
Average: 1190.82 ± 58

In [ ]:
!cat mzrc.log

mzrc00050439.fits 2026-08-20 16:05:50.760146    2.00 x16 -10 -10.0  39.8  33.8  33.9  22.9  28.6  1220.20   85.03   594 20441 18150 FOCUSING Shut FeNe Fan Cl1 Cl2
mzrc00050440.fits 2026-08-20 16:06:26.730974    2.00 x16 -10  -9.9  39.9  33.9  34.0  23.0  28.6  8594.50 16901.55   692 65535 17600 FOCUSING FeNe Fan Cl1 Cl2
mzrc00050441.fits 2026-08-20 16:06:30.015633    2.00 x16 -10  -9.9  39.9  33.9  34.0  22.9  28.6  8484.85 16775.49   584 65535 17700 FOCUSING FeNe Fan Cl1 Cl2
mzrc00050442.fits 2026-08-20 16:06:33.297504    2.00 x16 -10 -10.0  40.0  33.8  34.0  22.9  28.6  8327.61 16610.02   697 65535 17800 FOCUSING FeNe Fan Cl1 Cl2
mzrc00050443.fits 2026-08-20 16:06:36.594284    2.00 x16 -10 -10.0  40.0  33.8  34.0  22.9  28.6  8266.48 16638.42   709 65535 17900 FOCUSING FeNe Fan Cl1 Cl2
mzrc00050444.fits 2026-08-20 16:06:40.197125    2.00 x16 -10 -10.0  40.0  33.9  34.1  23.0  28.6  8075.52 16497.79   699 65535 18000 FOCUSING FeNe Fan Cl1 Cl2
mzrc00050445.fits 2026-08-20 16:06:43.8200

Reduce Target files (Ca)

In [ ]:
target = "V889_Her"
!grep {target} mzrc.log
!grep {target} mzrc.log > {target}_C

mzrc00067055.fits 2026-09-21 19:09:43.143201  150.00 x16 -20 -20.0  28.7  23.9  24.3  23.1  22.5  1549.31  801.64   656 65535 17997 V889_Her Fan
mzrc00067056.fits 2026-09-21 19:12:13.411488  150.00 x16 -20 -20.0  28.7  23.9  24.3  23.0  22.5  1557.22  830.20   671 65535 17997 V889_Her Fan
mzrc00067057.fits 2026-09-21 19:14:43.682527  150.00 x16 -20 -20.0  28.6  24.0  24.3  23.1  22.4  1555.04  805.28   552 65535 17997 V889_Her Fan
mzrc00067058.fits 2026-09-21 19:17:13.956554  150.00 x16 -20 -20.0  28.6  24.0  24.3  23.1  22.4  1542.16  753.73   659 65535 17997 V889_Her Fan
mzrc00067059.fits 2026-09-21 19:19:44.226021  150.00 x16 -20 -20.0  28.6  24.0  24.3  23.0  22.4  1544.60  769.59   679 65535 17997 V889_Her Fan
mzrc00067060.fits 2026-09-21 19:22:14.496858  150.00 x16 -20 -20.0  28.6  24.0  24.3  23.0  22.4  1553.04  802.41   690 65535 17997 V889_Her Fan
mzrc00067061.fits 2026-09-21 19:24:44.765295  150.00 x16 -20 -20.0  28.6  24.0  24.3  23.0  22.3  1547.10  778.25   631 65535 1799

In [ ]:
!../reduc {target}_C {target}_rC bias.fits flat.fits

Read: ../cal_C.dat ... OK
icen=605
Read bias.fits ... 2160 x 1280 format OK
Read ../ampgC1.fits ... 2160 x 1280 format OK
Read ../ampgC2.fits ... 2160 x 1280 format OK
Read ../ampgC3.fits ... 2160 x 1280 format OK
Read flat.fits  ... 2160 x 1280 format icen=604 OK
Read: V889_Her_C ... 
Read: V889_Her_C ... 
Read: mzrc00067055.fits ... gain=x16 (1) OK
N_CR=31  R_CRmax=3
Ampglow correction factor: -0.2012*ampgC1.fits+0.309117*ampgC2.fits-0.0154346*ampgC3.fits+28.5949
Bias correction
Write V889_Her_rC_001.fits ... 2160 x 1280 format OK
Read: mzrc00067056.fits ... gain=x16 (1) OK
N_CR=33  R_CRmax=7
Ampglow correction factor: -0.198274*ampgC1.fits+0.307181*ampgC2.fits-0.0159966*ampgC3.fits+30.985
Bias correction
Write V889_Her_rC_002.fits ... 2160 x 1280 format OK
Read: mzrc00067057.fits ... gain=x16 (1) OK
N_CR=32  R_CRmax=5
Ampglow correction factor: -0.199726*ampgC1.fits+0.30836*ampgC2.fits-0.0160891*ampgC3.fits+32.8589
Bias correction
Write V889_Her_rC_003.fits ... 2160 x 1280 format OK

In [ ]:
!rm spec_*.png {target}_C.log
!../double {target}_rC {target}_dC mzwarp_dxdy.fits anc

ストリーミング出力は最後の 5000 行に切り捨てられました。
(4253.871 120.326) => (4253.108 120.330)
(4253.526 602.008) => (4253.231 602.232)
V889_Her_rC_001.fits ... 2160 x 1280 format gain=x16 OK
av:245.969 sg:788.176
Write: double.bmp ... OK
Fiber flux (min/ave/max/bave): 0.0689986/921.17/5313.7/625.786
Sky level: 0.0426749
Fiber flux (min/ave/max/bave): 0.0689846/920.868/5312.99/625.558
Sky level: 0.0426763
Fiber flux (min/ave/max/bave): 0.0689847/920.868/5312.98/625.558
Sky level: 0.0426763
Kernel:  0.859415 0.933104 0.979646 1 0.997313 0.975263 0.928623 0.855291 0.801467
Fiber flux (min/ave/max/bave): 0.0689847/920.868/5312.98/625.558
Write double_im.fits ... 40 x 40 format OK
Write: double_y.dat ... OK
Center : -0.0658464 -0.0280311
Sigma :  2.52735
Fave  : 2198.05
S/N    : 100.558
Write: double_x.dat ... F_cont=2724.4+130.137*(w-394.425), EW_line=-1.95484 / -0.371908 OK
cp double_x.dat V889_Her_dC_001_x.dat
Write: double_im.bmp ... OK
Write double_rf.fits ... 4848 x 61 format OK
Write: double_rf.bmp ... O

In [ ]:
!convert -delay 50 -loop 0 spec_*.png {target}_C.gif
!rm spec_*.png
!../merge {target}_dC
!convert -delay 50 -loop 0 spec_*.png {target}_rC.gif

/bin/bash: line 1: convert: command not found
rm: cannot remove 'spec_*.png': No such file or directory
Read: V889_Her_dC_001_x.dat ... -0.065846 -0.028031 2.527350 2198.050049 OK
Read: V889_Her_dC_002_x.dat ... -0.062738 -0.032936 2.507180 2240.260010 OK
Read: V889_Her_dC_003_x.dat ... -0.080919 -0.030902 2.540330 2204.320068 OK
Read: V889_Her_dC_004_x.dat ... -0.085059 -0.042846 2.604950 2122.110107 OK
Read: V889_Her_dC_005_x.dat ... -0.072434 -0.022762 2.590730 2141.600098 OK
Read: V889_Her_dC_006_x.dat ... -0.062973 -0.019628 2.533110 2204.010010 OK
Read: V889_Her_dC_007_x.dat ... -0.084767 -0.044955 2.585090 2151.500000 OK
Read: V889_Her_dC_008_x.dat ... -0.075965 -0.044916 2.599520 2090.159912 OK
Read: V889_Her_dC_009_x.dat ... -0.104295 -0.015272 2.653990 1975.469971 OK
Read: V889_Her_dC_010_x.dat ... -0.104058 -0.024160 2.647830 2024.670044 OK
Read: V889_Her_dC_011_x.dat ... -0.083646 -0.018124 2.569220 2127.580078 OK
Read: V889_Her_dC_012_x.dat ... -0.062921 -0.040962 2.537750